# Physical AI Platform - Circuit Design-Space Lab

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**  
**License:** GNU General Public License v3.0-only (GPL-3.0-only). Third-party simulators, Python packages, PDKs and model libraries retain their own licenses.

This notebook accompanies the separate `drop-in/circuit-design-space/` module. It is an additional module beyond the ten folders in the Iteration 47 handoff. It is deliberately not nested inside `drop-in/circuit/`.

It demonstrates analytical RC and series-RLC responses, R×C and R×L×C design-space exploration, damping classification, constraints, RC parameter identification, identifiability, and optional JAX GPU execution. It does not run SPICE and does not make physical-verification claims.

**Open Google Colab:** [https://colab.research.google.com/](https://colab.research.google.com/). **Browser-only platform:** open this notebook through Google Colab. The public platform remains static-first; Python/SciPy/JAX run only in this explicitly started cloud runtime. Uploaded measurements/design data are processed by Google-hosted compute. Review confidentiality, IP, licensing, export and jurisdiction policy before upload.


## 0. Select an optional GPU runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** (availability depends on account/session). CPU mode remains fully supported. The notebook will report the actual JAX device. It will not call a CPU execution GPU.

In [ ]:
import sys, subprocess, importlib.util, shutil, zipfile, hashlib, json, math, os
from pathlib import Path
from google.colab import files
print(sys.version)
for name in ['numpy','scipy','jax']:
    print(name, bool(importlib.util.find_spec(name)))

## 1. Upload the separate circuit-design-space package ZIP

In [ ]:
uploaded = files.upload()
zip_candidates = [Path(n) for n in uploaded if n.lower().endswith('.zip')]
if not zip_candidates:
    raise RuntimeError('Upload circuit-design-space.zip')
selected = next((p for p in zip_candidates if p.name == 'circuit-design-space.zip'), zip_candidates[0])
root = Path('/content/pap-circuit-design-space')
if root.exists():
    import shutil as _shutil; _shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(selected) as zf:
    bad=zf.testzip()
    if bad: raise RuntimeError(f'ZIP CRC failure: {bad}')
    for info in zf.infolist():
        name=info.filename.replace('\\','/')
        parts=Path(name).parts
        if name.startswith('/') or '..' in parts or not parts:
            raise RuntimeError(f'Unsafe ZIP member path: {info.filename}')
        mode=(info.external_attr >> 16) & 0o170000
        if mode == 0o120000:
            raise RuntimeError(f'Symlink ZIP member rejected: {info.filename}')
        target=(root/name).resolve()
        if root.resolve() not in target.parents and target != root.resolve():
            raise RuntimeError(f'ZIP member escapes extraction root: {info.filename}')
        if info.is_dir(): target.mkdir(parents=True,exist_ok=True)
        else:
            target.parent.mkdir(parents=True,exist_ok=True)
            with zf.open(info) as src, target.open('wb') as dst:
                shutil.copyfileobj(src,dst)
module_files=list(root.rglob('module.py'))
if not module_files: raise RuntimeError('module.py not found')
MODDIR=module_files[0].parent
sys.path.insert(0,str(MODDIR))
print('Module directory:', MODDIR)

## 2. Verify the package SHA-256 manifest

The archive CRC is checked during extraction; this step verifies the individual package files against the manifest.

In [ ]:
manifest = MODDIR / 'MANIFEST.sha256'
if not manifest.exists(): raise RuntimeError('MANIFEST.sha256 missing')
checks=[]
for line in manifest.read_text().splitlines():
    if not line.strip(): continue
    expected, rel = line.split(None,1)
    rel=rel.strip().lstrip('*')
    target=(MODDIR/rel).resolve()
    if MODDIR.resolve() not in target.parents: raise RuntimeError(f'Unsafe manifest path: {rel}')
    checks.append((rel,hashlib.sha256(target.read_bytes()).hexdigest()==expected))
failed=[x for x in checks if not x[1]]
print(f'Integrity: {len(checks)-len(failed)}/{len(checks)} verified')
if failed: raise RuntimeError(f'Integrity failures: {failed}')

## 2. Install CPU dependencies and inspect JAX devices

JAX is optional. If JAX is absent, the NumPy CPU implementation remains available. For GPU execution, select a GPU runtime before importing JAX; use the current official JAX installation instructions if the Colab image does not include a compatible GPU build.

In [ ]:
missing = [name for name in ['numpy','scipy'] if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
import numpy as np
from scipy import __version__ as scipy_version
print('NumPy',np.__version__,'SciPy',scipy_version)
try:
    import jax
    print('JAX',jax.__version__)
    print('JAX devices:',[(d.platform,str(d)) for d in jax.devices()])
except Exception as exc:
    print('JAX not usable; CPU NumPy path only:',repr(exc))

## 3. Run the package test suite

In [ ]:
test = subprocess.run([sys.executable,'-m','unittest','-v'],cwd=MODDIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode: raise RuntimeError('Design-space tests failed')

## 4. Execute the shipped sample datasets

These are package-owned sample artifacts and are also covered by unit tests. The design-space grid and measurement identification run directly from the JSON files.

In [ ]:
from module import explore_rc_design_space, fit_rc_time_constant, explore_rlc_design_space
grid_sample=json.loads((MODDIR/'samples'/'rc-design-space.json').read_text())
measurement_sample=json.loads((MODDIR/'samples'/'rc-measurements.json').read_text())
sample_grid=explore_rc_design_space(**grid_sample['input'])
assert sample_grid['outcome']=='OK'
assert sample_grid['candidate_count']==grid_sample['expected']['candidate_count']
sample_fit=fit_rc_time_constant(measurement_sample['times_s'],measurement_sample['measured_voltage_v'],
    initial_v=measurement_sample['known_initial_v'],final_v=measurement_sample['known_final_v'])
assert sample_fit['outcome']=='OK'
assert abs(sample_fit['tau_s']-measurement_sample['expected_tau_s']) < 1e-8
print('Design grid sample PASS:',sample_grid['candidate_count'],'candidates')
print('Measurement sample PASS: tau=',sample_fit['tau_s'])
rlc_sample=json.loads((MODDIR/'samples'/'rlc-damping-regimes.json').read_text())
rlc_grid=explore_rlc_design_space(rlc_sample['resistance_ohm'],[rlc_sample['inductance_h']],[rlc_sample['capacitance_f']],rlc_sample['times_s'])
assert rlc_grid['outcome']=='OK'
assert [x['damping_regime'] for x in rlc_grid['candidates']]==rlc_sample['expected_regimes']
print('Series-RLC damping sample PASS:',rlc_sample['expected_regimes'])


## 4. Analytical RC reference and R×C design sweep

In [ ]:
from module import rc_step_response, explore_rc_design_space, fit_rc_time_constant, resistance_from_tau, local_sensitivity_tau

times = np.linspace(0.0001,0.010,100).tolist()
reference = rc_step_response(times, resistance_ohm=1000.0, capacitance_f=1e-6, step_v=5.0)
assert reference['outcome']=='OK'
print('tau (s):',reference['tau_s'],'first voltage (V):',reference['voltage_v'][0])

grid = explore_rc_design_space(
    resistances_ohm=[470, 1000, 2200, 4700],
    capacitances_f=[0.47e-6, 1e-6, 2.2e-6],
    times_s=times, step_v=5.0,
    backend='auto',
    constraints={'max_tau_s':0.005,'max_initial_current_a':0.02,'max_energy_j':0.0001}
)
assert grid['outcome']=='OK'
print('Engine:',grid['engine'],'candidates:',grid['candidate_count'],'feasible:',grid['feasible_count'])
for row in grid['candidates']:
    print(row['candidate_id'],row['R_ohm'],row['C_F'],row['tau_s'],row['feasible'],row['hold_reasons'])

## 5. Parameter identification and identifiability

In [ ]:
true_tau = 0.0025
measured = (1.0-np.exp(-np.asarray(times)/true_tau)).tolist()
fit = fit_rc_time_constant(times, measured, tau_bounds_s=(1e-5,0.1))
assert fit['outcome']=='OK'
print('Estimated tau:',fit['tau_s'],'RMSE:',fit['rmse_v'],'R²:',fit['r_squared'])
print('Identifiable:',fit['identifiable_parameters'])
print(fit['caveat'])
# Recover R only when C is independently known.
print(resistance_from_tau(fit['tau_s'], known_capacitance_f=1e-6))
print(local_sensitivity_tau(times, fit['tau_s'])['interpretation'])

## 6. Explicit GPU gate

In [ ]:
gpu_result = explore_rc_design_space([470,1000,2200],[0.47e-6,1e-6,2.2e-6],times,
                                     backend='jax',require_gpu=True)
if gpu_result['outcome']=='OK':
    assert gpu_result['engine']=='jax:gpu'
    print('GPU execution confirmed:',gpu_result['engine'])
else:
    print('GPU request correctly held:',gpu_result['reason'])
    print('No GPU claim is made. CPU results remain available with backend="numpy".')

## 7. Export design-space results

In [ ]:
outdir=Path('/content/pap-circuit-design-space-results')
outdir.mkdir(exist_ok=True)
(outdir/'design-space.json').write_text(json.dumps(grid,indent=2)+'\n')
(outdir/'parameter-fit.json').write_text(json.dumps(fit,indent=2)+'\n')
run_info={
 'module':'physical-ai-platform/drop-in/circuit-design-space',
 'module_version':'1.0.0',
 'engine':grid['engine'],
 'candidate_count':grid['candidate_count'],
 'feasible_count':grid['feasible_count'],
 'fit_parameter':'tau_s',
 'evidence_state':'UNVERIFIED',
 'physical_verification':'NOT_PERFORMED',
 'fidelity_promotion':'NOT_REQUESTED'
}
(outdir/'run-manifest.json').write_text(json.dumps(run_info,indent=2)+'\n')
import shutil as _shutil
archive=_shutil.make_archive('/content/circuit-design-space-results','zip',outdir)
files.download(archive)

## Copyright, attribution and license

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.

GNU General Public License v3.0-only (GPL-3.0-only). Third-party simulators, Python packages, PDKs and model libraries retain their own licenses.

NumPy, SciPy and JAX are third-party software under their respective licenses. This module does not include proprietary PDKs or foundry model data. Third-party model/library licenses are not changed by the package license.

**Engineering disclaimer:** the analytical RC model and parameter-fit outputs are not general circuit simulation, physical verification, hardware validation, safety approval, manufacturing signoff or certification. Results must be reviewed against the actual canonical design, measurement calibration and applicable requirements.